# 🎨 DesignBench — Google Colab Setup
**Benchmark for MLLM-based Front-end Code Generation**

https://github.com/WebPAI/DesignBench

---
Run cells **in order**, top to bottom. Each section is labeled clearly.

## 📦 CELL 1 — Clone Repo

In [1]:
import os

if not os.path.exists('/content/DesignBench'):
    !git clone https://github.com/WebPAI/DesignBench.git /content/DesignBench
else:
    print('✅ Repo already cloned, skipping.')

%cd /content/DesignBench
!mkdir -p data
!mkdir -p code/evaluator/res
!mkdir -p code/evaluator/tmp
print('✅ Repo ready.')

✅ Repo already cloned, skipping.
/content/DesignBench
✅ Repo ready.


In [2]:
# Patch mllm/__init__.py to use lazy imports (avoids needing mistralai, etc.)
patch = '''\
from .base import MLLMChat

__all__ = [
    "MLLMChat",
    "OpenAIChat",
    "AnthropicChat",
    "MistralChat",
    "GeminiChat",
    "DeepInfraChat",
    "QwenChat",
]

def get_model(model_name: str, **kwargs) -> MLLMChat:
    match model_name:
        case "gpt-4o-2024-11-20":
            from .openai_chat import OpenAIChat
            return OpenAIChat(model_name, **kwargs)
        case "claude-3-7-sonnet-20250219":
            from .anthropic_chat import AnthropicChat
            return AnthropicChat(model_name, **kwargs)
        case "gemini-2.0-flash":
            from .gemini_chat import GeminiChat
            return GeminiChat(model_name, **kwargs)
        case "qwen2.5-vl-72b-instruct" | "qwen2.5-vl-7b-instruct":
            from .platform_api import QwenChat
            return QwenChat(model_name, **kwargs)
        case "meta-llama/Llama-3.2-90B-Vision-Instruct" | "meta-llama/Llama-3.2-11B-Vision-Instruct":
            from .platform_api import DeepInfraChat
            return DeepInfraChat(model_name, **kwargs)
        case "pixtral-large-latest" | "pixtral-12b-2409":
            from .mistral_chat import MistralChat
            return MistralChat(model_name, **kwargs)
        case _:
            raise ValueError(f"Unsupported model name: {model_name}")
'''

with open('/content/DesignBench/code/mllm/__init__.py', 'w') as f:
    f.write(patch)
print('✅ Patched mllm/__init__.py for lazy imports')


✅ Patched mllm/__init__.py for lazy imports


In [3]:
# Patch QwenChat to use international API endpoint
import re

api_path = '/content/DesignBench/code/mllm/platform_api.py'
with open(api_path, 'r') as f:
    content = f.read()

content = content.replace(
    'https://dashscope.aliyuncs.com/compatible-mode/v1',
    'https://dashscope-intl.aliyuncs.com/compatible-mode/v1'
)

with open(api_path, 'w') as f:
    f.write(content)
print('✅ Patched QwenChat to use international endpoint')


✅ Patched QwenChat to use international endpoint


In [4]:
# Patch metric_utils.py to use Chrome instead of Firefox
path = '/content/DesignBench/code/evaluator/metric_utils.py'
with open(path, 'r') as f:
    content = f.read()

content = content.replace(
    'from selenium.webdriver.firefox.service import Service',
    'from selenium.webdriver.chrome.service import Service'
)
content = content.replace(
    'from selenium.webdriver.firefox.options import Options',
    'from selenium.webdriver.chrome.options import Options'
)
content = content.replace(
    'service = Service(executable_path=firefox_path)',
    'service = Service()'
)
content = content.replace(
    'driver = webdriver.Firefox(options=options, service=service)',
    'options.add_argument("--no-sandbox")\n        options.add_argument("--disable-dev-shm-usage")\n        driver = webdriver.Chrome(options=options, service=service)'
)
content = content.replace(
    "browser_name='firefox'",
    "browser_name='chrome'"
)


with open(path, 'w') as f:
    f.write(content)
print('✅ Patched metric_utils.py to use Chrome')


✅ Patched metric_utils.py to use Chrome


## 🌐 CELL 2 — Install Chrome + Firefox + Geckodriver

In [5]:
# ── Google Chrome ─────────────────────────────────────────────────
!apt-get update --fix-missing -qq
!apt-get install -f -qq
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb -qq
!google-chrome --version

# ── Firefox via Mozilla PPA (snap doesn't work in Colab) ──────────
!add-apt-repository ppa:mozillateam/ppa -y -q
!printf 'Package: *\nPin: release o=LP-PPA-mozillateam\nPin-Priority: 1001\n' > /etc/apt/preferences.d/mozilla-firefox
!apt-get update -qq
!apt-get install -y firefox -qq
!firefox --version

# ── Geckodriver ───────────────────────────────────────────────────
!wget -q https://github.com/mozilla/geckodriver/releases/download/v0.35.0/geckodriver-v0.35.0-linux64.tar.gz
!tar -xzf geckodriver-v0.35.0-linux64.tar.gz
!mv geckodriver /content/DesignBench/code/evaluator/geckodriver
!chmod +x /content/DesignBench/code/evaluator/geckodriver
!rm geckodriver-v0.35.0-linux64.tar.gz
!/content/DesignBench/code/evaluator/geckodriver --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Google Chrome 146.0.7680.164 
usage: add-apt-repository [-h] [-d] [-r] [-s] [-c COMPONENT] [-p POCKET] [-y]
                          [-n] [-l] [--dry-run] [-L] [-P PPA] [-C CLOUD]
                          [-U URI] [-S SOURCESLIST [SOURCESLIST ...]]
                          [line ...]
add-apt-repository: error: unrecognized arguments: -q
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

Command '/usr/bin/firefox' requires the firefox snap to be installed.
Please install it with:

snap install firefox

geckodriver 0.35.0 (9f0a0036bea4 2024-08-03 07:11 +0000)

The source code of this program is available from
testing/geckodriver in https://hg.mozilla.org/mozilla-

## 🟢 CELL 3 — Install Node v20 + single-file-cli + npm dependencies

In [6]:
# ── nvm + Node v20 ────────────────────────────────────────────────
!curl -fsSL https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.7/install.sh | bash -s -- --no-use
!bash -c 'source /root/.nvm/nvm.sh && nvm install 20 && nvm use 20 && node --version && npm --version'

# ── single-file-cli ───────────────────────────────────────────────
!bash -c 'source /root/.nvm/nvm.sh && nvm use 20 && npm install -g single-file-cli'
print('single-file-cli version:', end=' ')
!bash -c 'source /root/.nvm/nvm.sh && nvm use 20 && npx single-file --version'

# ── Web framework dependencies ────────────────────────────────────
print('\n📦 Installing React dependencies...')
!bash -c 'source /root/.nvm/nvm.sh && nvm use 20 && cd /content/DesignBench/web/my-react-app && npm install --silent'

print('📦 Installing Vue dependencies...')
!bash -c 'source /root/.nvm/nvm.sh && nvm use 20 && cd /content/DesignBench/web/my-vue-app && npm install --silent'

print('📦 Installing Angular dependencies...')
!bash -c 'source /root/.nvm/nvm.sh && nvm use 20 && npm install -g @angular/cli --silent && cd /content/DesignBench/web/my-angular-app && npm install --silent'

# ── Evaluator npm dependencies ────────────────────────────────────
print('📦 Installing evaluator dependencies...')
!bash -c 'source /root/.nvm/nvm.sh && nvm use 20 && cd /content/DesignBench/code/evaluator && npm install --silent'

print('\n✅ Node setup complete.')

=> nvm is already installed in /root/.nvm, trying to update using git
=> => Compressing and cleaning up git repository

=> nvm source string already in /root/.bashrc
=> bash_completion source string already in /root/.bashrc
=> Close and reopen your terminal to start using nvm or run the following to use it now:

export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && \. "$NVM_DIR/nvm.sh"  # This loads nvm
[ -s "$NVM_DIR/bash_completion" ] && \. "$NVM_DIR/bash_completion"  # This loads nvm bash_completion
v20.20.2 is already installed.
Now using node v20.20.2 (npm v10.8.2)
Now using node v20.20.2 (npm v10.8.2)
v20.20.2
10.8.2
Now using node v20.20.2 (npm v10.8.2)
⠙⠹⠸⠼⠴⠦⠧⠇
changed 3 packages in 1s
⠇single-file-cli version: Now using node v20.20.2 (npm v10.8.2)
⠙2.0.83
⠙
📦 Installing React dependencies...
Now using node v20.20.2 (npm v10.8.2)
📦 Installing Vue dependencies...
Now using node v20.20.2 (npm v10.8.2)
📦 Installing Angular dependencies...
Now using node v20.20.2 (npm v10.8.2)
📦 

## 🐍 CELL 4 — Install Python Packages

In [7]:
!pip install -q \
  anthropic==0.68.0 \
  openai==1.107.2 \
  google-generativeai==0.8.5 \
  google-api-python-client==2.181.0 \
  google-auth==2.40.3 \
  selenium==4.35.0 \
  opencv-python==4.11.0.86 \
  scikit-image==0.25.2 \
  pillow==11.3.0 \
  numpy \
  scipy==1.15.3 \
  openai-clip==1.0.1 \
  ftfy==6.3.1 \
  regex==2025.9.1 \
  tqdm==4.67.1 \
  requests==2.32.5 \
  retry==0.9.2 \
  imageio==2.37.0 \
  pydantic==2.11.9 \
  httpx==0.28.1 \
  filelock==3.19.1 \
  fsspec==2025.9.0 \
  networkx==3.4.2 \
  sympy==1.14.0 \
  jinja2==3.1.6 \
  packaging==25.0 \
  websocket-client==1.8.0 \
  trio==0.30.0 \
  trio-websocket==0.12.2 \
  python-dotenv

# torch separately in case version conflicts
!pip install -q torch torchvision

# verify
import torch, selenium, cv2, PIL, numpy
print('✅ Python packages installed.')
print(f'   torch: {torch.__version__}, numpy: {numpy.__version__}')

✅ Python packages installed.
   torch: 2.10.0+cu128, numpy: 2.4.4


## 💾 CELL 5 — Mount Google Drive & Extract Dataset

> Your dataset zip is named **`DesignBenchData.zip`** on Google Drive.

In [8]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, zipfile

zip_src = '/content/drive/MyDrive/designbench_data.zip'
data_dst = '/content/DesignBench/'

if not os.path.exists(zip_src):
    print(f'❌ Could not find {zip_src}')
    print('   Check your Drive — run: !ls "/content/drive/MyDrive/"')
else:
    print('📦 Extracting dataset...')
    with zipfile.ZipFile(zip_src, 'r') as z:
        z.extractall(data_dst)
    print(f'✅ Done! Contents of data/:')
    !ls /content/DesignBench/data/

Mounted at /content/drive
📦 Extracting dataset...
✅ Done! Contents of data/:
compile  edit  generation  repair


## ⚙️ CELL 6 — Configure config.py

In [9]:
import os, re, glob

# Find config.py anywhere in the repo
matches = glob.glob('/content/DesignBench/**/config.py', recursive=True)

if not matches:
    print('⚠️  config.py not found — listing all .py files to help locate it:')
    !find /content/DesignBench -name '*.py' | sort
else:
    for config_path in matches:
        with open(config_path, 'r') as f:
            content = f.read()

        content = re.sub(
            r'DesignBench_Path\s*=\s*["\'].*?["\']',
            'DesignBench_Path = "/content/DesignBench/"',
            content
        )

        with open(config_path, 'w') as f:
            f.write(content)

        print(f'✅ Patched: {config_path}')
        !grep 'DesignBench_Path' "{config_path}"

✅ Patched: /content/DesignBench/code/evaluator/config.py
DesignBench_Path = "/content/DesignBench/"
key_path = DesignBench_Path + "code/prompting/key.json"
firefox_path = DesignBench_Path + "code/evaluator/geckodriver"
    Task.GENERATION: DesignBench_Path + "data/DesignGeneration/",
    Task.EDIT: DesignBench_Path + "data/DesignEdit/",
    Task.REPAIR: DesignBench_Path + "data/DesignRepair/",
    Framework.VUE: DesignBench_Path + "web/my-vue-app/src/components/HelloWorld.vue",
    Framework.REACT: DesignBench_Path + "web/my-react-app/app/page.tsx",
    Framework.ANGULAR: DesignBench_Path + "web/my-angular-app/src/app/new.component.html"
    #     "html": DesignBench_Path + "web/my-angular-app/app/new.component.html",
    #     "ts": DesignBench_Path + "web/my-angular-app/app/new.component.ts"


## 🔐 CELL 7 — Set API Keys

Fill in the keys you have. Leave others as empty strings.

In [10]:
import os, json

# ── Fill in your keys here ────────────────────────────────────────
OPENAI_API_KEY      = ""   # gpt-4o etc.
ANTHROPIC_API_KEY   = ""   # claude
GEMINI_API_KEY      = ""   # gemini
QWEN_API_KEY        = "sk-2f3d520c69c64073a6f6fb6fbe51ebb4"   # qwen
DEEPINFRA_API_KEY   = ""   # llama via deepinfra
MISTRAL_API_KEY     = ""   # mistral / pixtral

# ── Write .env file ───────────────────────────────────────────────
env_content = f"""OPENAI_API_KEY="{OPENAI_API_KEY}"
ANTHROPIC_API_KEY="{ANTHROPIC_API_KEY}"
GEMINI_API_KEY="{GEMINI_API_KEY}"
QWEN_API_KEY="{QWEN_API_KEY}"
DEEPINFRA_API_KEY="{DEEPINFRA_API_KEY}"
MISTRAL_API_KEY="{MISTRAL_API_KEY}"
"""
with open('/content/DesignBench/.env', 'w') as f:
    f.write(env_content)

# ── Write code/prompting/key.json ─────────────────────────────────
key_json = {
    # "gpt":     OPENAI_API_KEY,
    # "claude":  ANTHROPIC_API_KEY,
    # "gemini":  GEMINI_API_KEY,
    "qwen":    QWEN_API_KEY
    # "llama":   DEEPINFRA_API_KEY,
    # "mistral": MISTRAL_API_KEY
}
os.makedirs('/content/DesignBench/code/prompting', exist_ok=True)
with open('/content/DesignBench/code/prompting/key.json', 'w') as f:
    json.dump(key_json, f, indent=2)

# ── Also set as env vars for this session ─────────────────────────
os.environ.update({
    'OPENAI_API_KEY':    OPENAI_API_KEY,
    'ANTHROPIC_API_KEY': ANTHROPIC_API_KEY,
    'GEMINI_API_KEY':    GEMINI_API_KEY,
    'QWEN_API_KEY':      QWEN_API_KEY,
    'DEEPINFRA_API_KEY': DEEPINFRA_API_KEY,
    'MISTRAL_API_KEY':   MISTRAL_API_KEY,
})

filled = [k for k, v in key_json.items() if v]
empty  = [k for k, v in key_json.items() if not v]
print(f'✅ Keys set:   {filled if filled else "none"}')
print(f'⚠️  Keys empty: {empty if empty else "none"}')

✅ Keys set:   ['qwen']
⚠️  Keys empty: none


## ✅ CELL 8 — Final Sanity Check

In [11]:
import subprocess, os

def check(name, cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or r.stderr).strip().split('\n')[0]
    icon = '✅' if r.returncode == 0 else '❌'
    print(f'{icon} {name:20s} {out}')

print('── Browsers & Drivers ──────────────────────────────')
check('Chrome',        'google-chrome --version')
check('Firefox',       'firefox --version')
check('Geckodriver',   '/content/DesignBench/code/evaluator/geckodriver --version')

print('\n── Node & npm tools ────────────────────────────────')
check('Node',          "bash -c 'source /root/.nvm/nvm.sh && node --version'")
check('npm',           "bash -c 'source /root/.nvm/nvm.sh && npm --version'")
check('single-file',   "bash -c 'source /root/.nvm/nvm.sh && nvm use 20 -s && npx single-file --version'")

print('\n── Python packages ─────────────────────────────────')
for pkg in ['torch', 'selenium', 'cv2', 'PIL', 'numpy', 'anthropic', 'openai']:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', 'ok')
        print(f'✅ {pkg:20s} {ver}')
    except ImportError:
        print(f'❌ {pkg:20s} NOT FOUND')

print('\n── Paths & Files ───────────────────────────────────')
paths = [
    '/content/DesignBench/data',
    '/content/DesignBench/.env',
    '/content/DesignBench/code/prompt/key.json',
    '/content/DesignBench/code/evaluator/res',
    '/content/DesignBench/code/evaluator/tmp',
]
for p in paths:
    exists = os.path.exists(p)
    icon = '✅' if exists else '❌'
    extra = f'({len(os.listdir(p))} items)' if exists and os.path.isdir(p) else ''
    print(f'{icon} {p} {extra}')

import os
print(os.getenv("QWEN_API_KEY"))
with open('/content/DesignBench/code/mllm/platform_api.py', 'r') as f:
    for line in f:
        if 'dashscope' in line:
            print(line.strip())
print(repr(os.getenv("QWEN_API_KEY")))



── Browsers & Drivers ──────────────────────────────
✅ Chrome               Google Chrome 146.0.7680.164
❌ Firefox              Command '/usr/bin/firefox' requires the firefox snap to be installed.
✅ Geckodriver          geckodriver 0.35.0 (9f0a0036bea4 2024-08-03 07:11 +0000)

── Node & npm tools ────────────────────────────────
✅ Node                 v20.20.2
✅ npm                  10.8.2
❌ single-file          N/A: version "-s" is not yet installed.

── Python packages ─────────────────────────────────
✅ torch                2.10.0+cu128
✅ selenium             4.35.0
✅ cv2                  4.11.0
✅ PIL                  11.3.0
✅ numpy                2.4.4
✅ anthropic            0.68.0
✅ openai               1.107.2

── Paths & Files ───────────────────────────────────
✅ /content/DesignBench/data (5 items)
✅ /content/DesignBench/.env 
✅ /content/DesignBench/code/prompt/key.json 
✅ /content/DesignBench/code/evaluator/res (0 items)
✅ /content/DesignBench/code/evaluator/tmp (0 items)
sk-

In [12]:
from openai import OpenAI

client = OpenAI(
    api_key="sk-2f3d520c69c64073a6f6fb6fbe51ebb4",
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
)

response = client.chat.completions.create(
    model="qwen2.5-vl-72b-instruct",
    messages=[{"role": "user", "content": "hello"}],
    max_tokens=10,
)
print(response.choices[0].message.content)


Hello! How can I assist you today?


---
## 🚀 CELL 9 — Run: Generation Task

Edit the model, framework, and settings below then run.

In [13]:
!ls /content/DesignBench/code/runner/


__init__.py  main.py  __pycache__


In [14]:
import sys
sys.path.insert(0, '/content/DesignBench/code')
print(sys.path[:3])

import importlib
importlib.invalidate_caches()

from runner.main import Runner


['/content/DesignBench/code', '/content', '/env/python']


In [15]:
import sys
sys.path.insert(0, '/content/DesignBench/code')

from runner.main import Runner
from utils import Framework, Task, Mode

runner = Runner(
    "qwen2.5-vl-72b-instruct",
    framework=Framework.REACT,
    stream=True,
    print_content=True,
)


runner.run(
    task=Task.REPAIR,
    output_framework=Framework.REACT,
    mode=Mode.BOTH,
    max_workers=1,
    execution_range=(1, 2),      # runs sample 1 only (range is exclusive)
)


Temperature: 0, Max Tokens: 8192, Seed: 42


qwen2.5-vl-72b-instruct: Running repair (react -> react / both):   0%|          | 0/1 [00:00<?, ?it/s]

## 📊 CELL 10 — Evaluate: Generation Task

In [16]:
import sys
sys.path.insert(0, '/content/DesignBench/code')

from evaluator.main import *
from evaluator.compile import *

models = [
    "gemini-2.0-flash",
    # "gpt-4o-2024-11-20",
    # "claude-3-7-sonnet-20250219",
    # "qwen2.5-vl-72b-instruct",
]

frame_works             = ["react", "vue", "angular", "vanilla"]
implemented_frame_works = ["react", "vue", "angular", "vanilla"]

# Run evaluation
evaluate_generation(models=models, frame_works=frame_works, implemented_frameworks=implemented_frame_works)

# Collect compile info (skip vanilla — it doesn't compile)
for fw in frame_works:
    if fw == "vanilla":
        continue
    for impl in implemented_frame_works:
        collect_compile_information(
            task_name=Task.GENERATION,
            frame_work=fw,
            implemented_framework_or_mode=impl
        )

ModuleNotFoundError: No module named 'metric'

## 📊 CELL 11 — Evaluate: Edit Task

In [ ]:
import sys
sys.path.insert(0, '/content/DesignBench/code')

from evaluator.main import *
from evaluator.compile import *

models = [
    "gemini-2.0-flash",
    # add more models here
]

frame_works = ["react", "vue", "angular", "vanilla"]
modes       = ["both", "code", "image"]

evaluate_edit(models=models, frame_works=frame_works, modes=modes, llm_judge_flag=False)
evaluate_edit(models=models, frame_works=frame_works, modes=modes, llm_judge_flag=True)

for fw in frame_works:
    if fw == "vanilla":
        continue
    for mode in modes:
        collect_compile_information(
            task_name=Task.EDIT,
            frame_work=fw,
            implemented_framework_or_mode=mode
        )

## 📊 CELL 12 — Evaluate: Repair Task

In [19]:
import subprocess
subprocess.Popen(
    'bash -c "source /root/.nvm/nvm.sh && nvm use 20 && cd /content/DesignBench/web/my-react-app && npm run dev"',
    shell=True,
    stdout=open('/tmp/react-dev.log', 'w'),
    stderr=subprocess.STDOUT
)

import time
time.sleep(15)
!curl -s http://localhost:3000 | head -1


<!DOCTYPE html><html lang="en"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="preload" as="image" href="https://placehold.co/100x100"/><link rel="stylesheet" href="/_next/static/css/app/layout.css?v=1774912054521" data-precedence="next_static/css/app/layout.css"/><link rel="preload" as="script" fetchPriority="low" href="/_next/static/chunks/webpack.js?v=1774912054521"/><script src="/_next/static/chunks/main-app.js?v=1774912054521" async=""></script><script src="/_next/static/chunks/app-pages-internals.js" async=""></script><script src="/_next/static/chunks/app/page.js" async=""></script><title>Create Next App</title><meta name="description" content="Generated by create next app"/><link rel="icon" href="/favicon.ico" type="image/x-icon" sizes="16x16"/><meta name="next-size-adjust"/><script src="/_next/static/chunks/polyfills.js" noModule=""></script></head><body class="__variable_1e4310 __variable_c3aa02 antialiased"><div cla

In [29]:
import os
!mkdir -p /content/DesignBench/tmp

print(os.getcwd())
!ls -d ./tmp
!bash -c 'source /root/.nvm/nvm.sh && which node'


/content/DesignBench
./tmp
/root/.nvm/versions/node/v20.20.2/bin/node


In [30]:
path = '/content/DesignBench/code/evaluator/metric_ast.py'
with open(path, 'r') as f:
    content = f.read()

content = content.replace(
    '/Users/whalexiao/.nvm/versions/node/v18.19.0/bin/node',
    '/root/.nvm/versions/node/v20.20.2/bin/node'
)

with open(path, 'w') as f:
    f.write(content)
print('✅ Patched metric_ast.py node path')


✅ Patched metric_ast.py node path


In [40]:

!ln -sf /content/DesignBench/data/repair/* /content/DesignBench/data/DesignRepair/
import importlib, metric_ast
importlib.reload(metric_ast)
!bash -c 'source /root/.nvm/nvm.sh && nvm use 20 && npm install @babel/parser --prefix /content/DesignBench'

import os
results_dir = '/content/DesignBench/data/DesignRepair/RepairResults/react-react/qwen2.5-vl-72b-instruct/'
existing = [f for f in os.listdir(results_dir) if f.endswith('.jsx')]
print(f"Found {len(existing)} result files: {existing}")

import sys
sys.path.insert(0, '/content/DesignBench/code')
sys.path.insert(0, '/content/DesignBench/code/evaluator')

import evaluator.main
evaluator.main.re_calculate = False

!mkdir -p /content/DesignBench/data/DesignRepair/RepairResults
!ln -sf /content/DesignBench/results/repair/* /content/DesignBench/data/DesignRepair/RepairResults/

from evaluator.main import *
from evaluator.compile import *

models = ["qwen2.5-vl-72b-instruct"]
frame_works = ["react"]
modes = ["both"]
from evaluator.main import get_repair_metric

metric = get_repair_metric(
    web_name="1",
    model_name="qwen2.5-vl-72b-instruct",
    framework="react",
    mode="both",
    llm_judge_flag=False
)
print(metric)


Now using node v20.20.2 (npm v10.8.2)
⠙⠹⠸⠼
added 4 packages in 807ms
⠼Found 1 result files: ['react_1_qwen2.5-vl-72b-instruct_react_both.jsx']
/content/DesignBench/data/DesignRepair/RepairResults/react-react/qwen2.5-vl-72b-instruct/react_1_qwen2.5-vl-72b-instruct_react_both.jsx
Parsing before_code...
Parsing gt_code...
Parsing pred_code...
Extracting GT operations...
Found 42 GT operations
Extracting pred operations...
Found 22 pred operations
Matching operations...
Matched 9 operation pairs
GT matched content length: 448
Pred matched content length: 693
Calculating BLEU on matched operations...
Calculating CodeBLEU on matched operations...

AST Edit Similarity Results (CodeBLEU on Matched Operations)
  AST-TED (Tree Edit Distance):     0.9830
  AST-OP (Operation Matching):      0.2143
    - Matched operations:            9
    - Total GT operations:           42
    - Total Pred operations:         22
----------------------------------------------------------------------
BLEU Scores (

In [37]:
!ln -sf /content/DesignBench/data/repair/* /content/DesignBench/data/DesignRepair/
import importlib, metric_ast
importlib.reload(metric_ast)
!bash -c 'source /root/.nvm/nvm.sh && nvm use 20 && npm install -g @babel/parser'

import os
results_dir = '/content/DesignBench/data/DesignRepair/RepairResults/react-react/qwen2.5-vl-72b-instruct/'
existing = [f for f in os.listdir(results_dir) if f.endswith('.jsx')]
print(f"Found {len(existing)} result files: {existing}")

import sys
sys.path.insert(0, '/content/DesignBench/code')
sys.path.insert(0, '/content/DesignBench/code/evaluator')

import evaluator.main
evaluator.main.re_calculate = False

!mkdir -p /content/DesignBench/data/DesignRepair/RepairResults
!ln -sf /content/DesignBench/results/repair/* /content/DesignBench/data/DesignRepair/RepairResults/

from evaluator.main import *
from evaluator.compile import *

models = ["qwen2.5-vl-72b-instruct"]
frame_works = ["react"]
modes = ["both"]

evaluate_repair(models=models, frame_works=frame_works, modes=modes, llm_judge_flag=False)


Now using node v20.20.2 (npm v10.8.2)
⠙⠹⠸
changed 4 packages in 540ms
⠸Found 1 result files: ['react_1_qwen2.5-vl-72b-instruct_react_both.jsx']


  0%|          | 0/28 [00:00<?, ?it/s]

/content/DesignBench/data/DesignRepair/RepairResults/react-react/qwen2.5-vl-72b-instruct/react_1_qwen2.5-vl-72b-instruct_react_both.jsx
Parsing before_code...
Error calculating AST similarity: Parser error: node:internal/modules/cjs/loader:1210
  throw err;
  ^

Error: Cannot find module '@babel/parser'
Require stack:
- /content/DesignBench/tmp/tmpuk628w99/parser.js
    at Module._resolveFilename (node:internal/modules/cjs/loader:1207:15)
    at Module._load (node:internal/modules/cjs/loader:1038:27)
    at Module.require (node:internal/modules/cjs/loader:1289:19)
    at require (node:internal/modules/helpers:182:18)
    at Object.<anonymous> (/content/DesignBench/tmp/tmpuk628w99/parser.js:2:15)
    at Module._compile (node:internal/modules/cjs/loader:1521:14)
    at Module._extensions..js (node:internal/modules/cjs/loader:1623:10)
    at Module.load (node:internal/modules/cjs/loader:1266:32)
    at Module._load (node:internal/modules/cjs/loader:1091:12)
    at Function.executeUserEntr

Traceback (most recent call last):
  File "/content/DesignBench/code/evaluator/metric_ast.py", line 576, in calculate
    return self._full_ast_similarity(before_code, gt_code, pred_code, file_type)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/DesignBench/code/evaluator/metric_ast.py", line 610, in _full_ast_similarity
    before_ast = self.parser.parse(before_code, file_type)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/DesignBench/code/evaluator/metric_ast.py", line 358, in parse
    raise Exception(f"Parser error: {result.stderr}")
Exception: Parser error: node:internal/modules/cjs/loader:1210
  throw err;
  ^

Error: Cannot find module '@babel/parser'
Require stack:
- /content/DesignBench/tmp/tmpuk628w99/parser.js
    at Module._resolveFilename (node:internal/modules/cjs/loader:1207:15)
    at Module._load (node:internal/modules/cjs/loader:1038:27)
    at Module.require (node:internal/modules/cjs/lo

{'occlusion', 'alignment'}
{'MAE': np.float64(65.43500478316327), 'clip_similarity': 0.8642578125, 'structure_similarity': np.float64(0.8323549149923518), 'code_score': 0.2727272727272727, 'issue accuracy': 0.5, 'ast_code_op_score': 0, 'ast_code_content_score': 0, 'ast_code_content_weighted_score': 0}
/content/DesignBench/data/DesignRepair/RepairResults/react-react/qwen2.5-vl-72b-instruct/react_2_qwen2.5-vl-72b-instruct_react_both.jsx


FileNotFoundError: [Errno 2] No such file or directory: '/content/DesignBench/data/DesignRepair/RepairResults/react-react/qwen2.5-vl-72b-instruct/react_2_qwen2.5-vl-72b-instruct_react_both.jsx'

In [ ]:
!find /content/DesignBench/results -name "*qwen*" -type f


In [ ]:
!google-chrome --version
!which chromedriver


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
driver.get("https://example.com")
print(driver.title)
driver.quit()


In [ ]:
!grep -n "webdriver.Chrome\|webdriver.Firefox" /content/DesignBench/code/evaluator/metric_utils.py


In [ ]:
import subprocess
subprocess.Popen(
    'bash -c "source /root/.nvm/nvm.sh && nvm use 20 && cd /content/DesignBench/web/my-react-app && npm run dev"',
    shell=True,
    stdout=open('/tmp/react-dev.log', 'w'),
    stderr=subprocess.STDOUT
)

import time
time.sleep(15)
!curl -s http://localhost:3000 | head -5


In [ ]:
!grep "browser_name=" /content/DesignBench/code/evaluator/metric_utils.py
